In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['cybergis_job-supervisor',
            'libfive_libfive',
            'hpi-information-systems_metanome',
            'nasa_smcpy',
            'brownhci_drafty',
            'andergray_probabilityboundsanalysis.jl',
            'computationalmodelling_fidimag',
            'wsshin_maxwellfdfd',
            'pytorch_fbgemm',
            'stardist_stardist']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  #prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0075ff18028044c525d5d6dfc17926081b3a889b,cybergis_job-supervisor


In [2]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  5%|▌         | 158/3145 [02:53<53:53,  1.08s/it] 

Got Errors {'16d36be32a171fdd0b5f5b3bcf967099af846fc1': 'Key 16d36be32a171fdd0b5f5b3bcf967099af846fc1 not found in /da5_fast/All.sha1c/commit_22.tch'}


 18%|█▊        | 579/3145 [10:31<49:37,  1.16s/it]

Got Errors {'394cb53e1f389dca891822832823060ba9f9805a': 'Key 394cb53e1f389dca891822832823060ba9f9805a not found in /da5_fast/All.sha1c/commit_57.tch'}


 20%|█▉        | 619/3145 [11:16<45:47,  1.09s/it]

...

Got Errors {'dabf57702a91860224567c6aea06e3d5ac47adf3': 'Key dabf57702a91860224567c6aea06e3d5ac47adf3 not found in /da5_fast/All.sha1c/commit_90.tch'}


 93%|█████████▎| 2921/3145 [53:12<04:05,  1.09s/it]

Got Errors {'f147b0c10ab3fcc3417c6cafbb08724b0647e4c9': 'Key f147b0c10ab3fcc3417c6cafbb08724b0647e4c9 not found in /da5_fast/All.sha1c/commit_113.tch'}


100%|██████████| 3145/3145 [57:17<00:00,  1.09s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0075ff18028044c525d5d6dfc17926081b3a889b,4fb25a249e469352cf8e85bc8f988f1ca1d68c30,[2495fdcf6ed4d2ea6692b9cdd7792f9851d2400f],Alexander Michels <michels9@illinois.edu>,1750783227,-0500,GitHub <noreply@github.com>,1750783227,-0500,speed up git fetch and updated containers to u...,0075ff18028044c525d5d6dfc17926081b3a889b,cybergis_job-supervisor
1,010a5b360d94b6c978c08452b9c41e24e9d46dd4,c685b1b3547e512d9dab7d6d9792a03bd748a1db,[d284286f67d287539a1f15806f7664431f7e3908],Ian Zhang <yianzhang14@gmail.com>,1740535063,-0600,Ian Zhang <yianzhang14@gmail.com>,1740535063,-0600,more syncing\n,010a5b360d94b6c978c08452b9c41e24e9d46dd4,cybergis_job-supervisor


In [3]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0075ff18028044c525d5d6dfc17926081b3a889b', 'tree': '4fb25a249e469352cf8e85bc8f988f1ca1d68c30', 'parent': ['2495fdcf6ed4d2ea6692b9cdd7792f9851d2400f'], 'author': 'Alexander Michels <michels9@illinois.edu>', 'author_time': 1750783227, 'author_tz': '-0500', 'committer': 'GitHub <noreply@github.com>', 'committer_time': 1750783227, 'committer_tz': '-0500', 'message': 'speed up git fetch and updated containers to use new path on anvil (#156)\n\nCo-authored-by: mitkotak <zyli2004@gmail.com>'}


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [8]:
netid = 'hcrettol'

# Save all the commits to one csv file
out = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']]
out.to_csv(netid + '_project_summary.csv', sep=';', index=False, header=False)
print('Saved', len(out), 'commits')

# Count commits and authors for each project, and find the first and last commit
stats = out.groupby('project').agg(
    ncommits=('commit', 'count'),
    nauthors=('author', 'nunique'),
    min_time=('author_time', 'min'),
    max_time=('author_time', 'max'))

# Turn the timestamps into normal dates
stats['from'] = pd.to_datetime(stats['min_time'], unit='s').dt.strftime('%Y-%m-%d')
stats['to'] = pd.to_datetime(stats['max_time'], unit='s').dt.strftime('%Y-%m-%d')
print(stats)

Saved 31421 commits
                                        ncommits  nauthors    min_time  \
project                                                                  
andergray_probabilityboundsanalysis.jl       406         8  1566923172   
brownhci_drafty                             2490        21  1454018185   
computationalmodelling_fidimag              2007        45  1354627521   
cybergis_job-supervisor                     1198        27  1594862452   
hpi-information-systems_metanome            2834        54  1363423355   
libfive_libfive                             3975       114  1448854231   
nasa_smcpy                                  1064        28  1536353601   
pytorch_fbgemm                             15766       852  1539384493   
stardist_stardist                           1312        41  1530047944   
wsshin_maxwellfdfd                           369         4  1351309510   

                                          max_time        from          to  
project       

| Project | Commits | Authors | From | To | Stars | Forks | Last GH Commit |
|---|---|---|---|---|---|---|---|
| andergray_probabilityboundsanalysis.jl | 406 | 8 | 2019-08-27 | 2025-03-10 | 26 | 2 | 2025-01-13 |
| brownhci_drafty | 2490 | 21 | 2016-01-28 | 2025-08-01 | 9 | 4 | 2025-05-29 |
| computationalmodelling_fidimag | 2007 | 45 | 2012-12-04 | 2025-10-17 | 68 | 27 | 2026-09-28 |
| cybergis_job-supervisor | 1198 | 27 | 2020-07-16 | 2025-10-31 | 12 | 5 | 2025-06-24 |
| hpi-information-systems_metanome | 2834 | 54 | 2013-03-16 | 2025-08-25 | 194 | 68 | 2025-06-05 |
| libfive_libfive | 3975 | 114 | 2015-11-30 | 2025-08-19 | 1700 | 177 | 2025-11-12 |
| nasa_smcpy | 1064 | 28 | 2018-09-07 | 2025-09-16 | 137 | 37 | 2026-07-24 |
| pytorch_fbgemm | 15766 | 852 | 2018-10-12 | 2025-11-06 | 1600 | 790 | 2026-09-28 |
| stardist_stardist | 1312 | 41 | 2018-06-26 | 2025-07-31 | 1300 | 283 | 2026-02-14 |
| wsshin_maxwellfdfd | 369 | 4 | 2012-10-27 | 2025-04-16 | 123 | 51 | 2025-04-15 |